# Leer un modelo de difusión y generar con él

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento Digital de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Aprender a leer la ficha y la estructura de un modelo en Hugging Face, y a controlar qué genera.

## Al terminar este cuaderno vas a poder:

1. Leer licencia, tarea y componentes de un modelo antes de usarlo.
2. Generar una imagen con `diffusers` y explicar `num_inference_steps`, `guidance_scale` y `seed`.
3. Medir cuánto tardan y cuánta memoria usan los modelos.

## Preparación

Este cuaderno necesita **GPU**: en Colab, *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

`transformers` aporta el modelo que entiende el texto, y `accelerate` ayuda a cargar los modelos grandes de forma eficiente.

In [ ]:
%pip install -q diffusers transformers accelerate

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU. Menú: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU')

print('GPU:', torch.cuda.get_device_name(0))
print('Memoria de la GPU:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

## 1. Leer la ficha de un modelo

Antes de usar un modelo conviene leer su ficha en [huggingface.co/stabilityai/sd-turbo](https://huggingface.co/stabilityai/sd-turbo). Desde Python podemos pedir los datos principales.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
nombre_del_modelo = 'stabilityai/sd-turbo'
ficha = api.model_info(nombre_del_modelo)

print('Tarea:', ficha.pipeline_tag)  # qué hace el modelo
print('Biblioteca:', ficha.library_name)  # con qué se usa
print('Licencia en los metadatos:', ficha.card_data.license or 'no declarada')  # qué podés hacer con él
print('Pide aceptar condiciones:', ficha.gated)
print('Descargas recientes:', ficha.downloads)

Este modelo **no declara la licencia en los metadatos**. Cuando pasa eso hay que leer el texto de la ficha. Buscamos las líneas que nombran la licencia:

In [ ]:
from huggingface_hub import ModelCard

texto = ModelCard.load(nombre_del_modelo).text  # el texto completo de la ficha
for linea in texto.splitlines():
    if 'licen' in linea.lower():
        print('-', linea.strip())

> **Para mirar.** La ficha remite a la licencia de Stability AI para el uso comercial. Para un TPI o un proyecto personal alcanza con citar el modelo; para un producto, hay que leer esa licencia.

Un modelo de `diffusers` es una **carpeta de piezas**. Cada subcarpeta es un componente:

In [ ]:
archivos = api.list_repo_files(nombre_del_modelo)
carpetas = sorted({a.split('/')[0] for a in archivos if '/' in a})
print('Piezas del modelo:', carpetas)

ruta = hf_hub_download(nombre_del_modelo, 'model_index.json')  # el "índice" del modelo
print(open(ruta).read())

`model_index.json` dice qué clase usa cada pieza. Las tres importantes:

- `text_encoder`: convierte el *prompt* en números.
- `unet`: quita el ruido (es lo que viste en el cuaderno 01).
- `vae`: pasa entre píxeles y el espacio comprimido donde trabaja la UNet.

## 2. Cargar el modelo

`SD-Turbo` es un modelo **destilado**: aprendió a generar en 1 a 4 pasos en lugar de 30 a 50.

- `torch_dtype=torch.float16`: usa números de 16 bits; la mitad de memoria y más velocidad.
- `variant='fp16'`: descarga la versión ya guardada en 16 bits.
- `.to('cuda')`: mueve el modelo a la GPU.

La primera vez descarga unos 2 GB (1 a 3 minutos).

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    nombre_del_modelo,
    torch_dtype=torch.float16,
    variant='fp16',
).to('cuda')

for nombre, pieza in pipe.components.items():
    if hasattr(pieza, 'parameters'):
        millones = sum(p.numel() for p in pieza.parameters()) / 1e6
        print(f'{nombre:<18} {millones:>7.1f} millones de parámetros')
    else:
        print(f'{nombre:<18} (sin parámetros)')

## 3. Primera imagen

Parámetros de la llamada:

- `prompt`: la descripción. Los modelos funcionan mejor en inglés.
- `num_inference_steps`: cuántos pasos de limpieza (en Turbo alcanzan 1 a 4).
- `guidance_scale`: cuánto empuja el texto. Turbo se entrenó **sin** esta guía: se usa `0.0`.
- `generator`: fija la *seed*, es decir, el ruido inicial.

In [ ]:
import matplotlib.pyplot as plt

prompt = 'a photograph of an old wooden house in a village, soft morning light'
generador = torch.Generator('cuda').manual_seed(42)

imagen = pipe(
    prompt=prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
    generator=generador,
).images[0]

plt.figure(figsize=(5, 5))
plt.imshow(imagen)
plt.axis('off')
plt.show()

## 4. Cuántos pasos

Misma *seed*, distinta cantidad de pasos. Más pasos refinan el detalle, pero con Turbo la mejora es chica.

In [ ]:
lista_de_pasos = [1, 2, 4]

figura, ejes = plt.subplots(1, len(lista_de_pasos), figsize=(15, 5))
for numero in range(len(lista_de_pasos)):
    generador = torch.Generator('cuda').manual_seed(42)  # misma seed en todas
    imagen = pipe(prompt, num_inference_steps=lista_de_pasos[numero], guidance_scale=0.0, generator=generador).images[0]
    ejes[numero].imshow(imagen)
    ejes[numero].set_title(f'{lista_de_pasos[numero]} paso(s)')
    ejes[numero].axis('off')
plt.show()

## 5. Qué hace la seed

Mismo prompt, distinta *seed*: otro ruido inicial, otra imagen. Por eso guardar la *seed* hace reproducible un resultado.

In [ ]:
semillas = [1, 2, 3, 4]

figura, ejes = plt.subplots(1, len(semillas), figsize=(18, 4.5))
for numero in range(len(semillas)):
    generador = torch.Generator('cuda').manual_seed(semillas[numero])
    imagen = pipe(prompt, num_inference_steps=1, guidance_scale=0.0, generator=generador).images[0]
    ejes[numero].imshow(imagen)
    ejes[numero].set_title(f'seed = {semillas[numero]}')
    ejes[numero].axis('off')
plt.show()

## 6. Qué pasa con `guidance_scale`

En modelos comunes (Stable Diffusion 1.5) se usa un valor de 5 a 8. En Turbo no: probamos tres valores.

In [ ]:
guias = [0.0, 1.0, 3.0]

figura, ejes = plt.subplots(1, len(guias), figsize=(15, 5))
for numero in range(len(guias)):
    generador = torch.Generator('cuda').manual_seed(42)
    imagen = pipe(prompt, num_inference_steps=2, guidance_scale=guias[numero], generator=generador).images[0]
    ejes[numero].imshow(imagen)
    ejes[numero].set_title(f'guidance_scale = {guias[numero]}')
    ejes[numero].axis('off')
plt.show()

> **Para mirar.** Con valores altos suele aparecer saturación o artefactos: el modelo no fue entrenado para eso. Cada familia de modelos tiene sus valores recomendados, y están en su ficha.

## 7. Tiempo y memoria

Dos números que conviene conocer antes de elegir un modelo: cuánto tarda y cuánta memoria de la GPU usa. Probamos con dos tamaños de imagen. SD-Turbo se entrenó con **512 × 512**.

In [ ]:
import time

tamanios = [512, 768]

figura, ejes = plt.subplots(1, len(tamanios), figsize=(12, 6))
for numero in range(len(tamanios)):
    torch.cuda.reset_peak_memory_stats()  # reiniciamos el contador de memoria
    inicio = time.time()
    generador = torch.Generator('cuda').manual_seed(42)
    imagen = pipe(
        prompt,
        height=tamanios[numero],
        width=tamanios[numero],
        num_inference_steps=1,
        guidance_scale=0.0,
        generator=generador,
    ).images[0]
    torch.cuda.synchronize()  # esperamos a que la GPU termine
    segundos = time.time() - inicio
    memoria = torch.cuda.max_memory_allocated() / 1024**3
    ejes[numero].imshow(imagen)
    ejes[numero].set_title(f'{tamanios[numero]} px: {segundos:.1f} s, {memoria:.1f} GB')
    ejes[numero].axis('off')
plt.show()

Cuando un modelo no entra en la memoria hay cuatro salidas, de menor a mayor esfuerzo:

1. **Achicar la imagen** antes de procesarla.
2. Usar `float16` (ya lo hicimos).
3. Llamar a `pipe.enable_model_cpu_offload()` para que las piezas esperen en la CPU cuando no se usan.
4. Elegir un modelo más chico.

Al terminar con un modelo, liberamos la memoria para poder cargar el siguiente:

In [ ]:
import gc

del pipe  # soltamos el modelo
gc.collect()
torch.cuda.empty_cache()
print('Memoria en uso:', round(torch.cuda.memory_allocated() / 1024**3, 2), 'GB')

> **Para probar.** Cambiá el `prompt` por uno en español y compará. Después describí la misma escena con más y con menos detalle. Anotá qué cambia y qué no.

## Glosario

| Término | Qué significa acá |
|---|---|
| **Pipeline** | Objeto que reúne todas las piezas del modelo y las ejecuta en orden. |
| **Prompt** | Texto que describe lo que querés generar. |
| **Destilación** | Entrenar un modelo para imitar a otro más lento pero en muy pocos pasos. |
| **`num_inference_steps`** | Cantidad de pasos de limpieza al generar. |
| **`guidance_scale`** | Cuánto empuja el texto sobre el resultado. |
| **`float16`** | Números de 16 bits: menos memoria y más velocidad. |
| **VRAM** | Memoria de la GPU. |